<a href="https://colab.research.google.com/github/Angelin-ak/Micro_project1_Campus-City-Emergency-Supply-Distribution/blob/main/micro_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
from google.colab import files
import pandas as pd
import io
import pulp
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# --- STEP 1: REQUEST FILES FROM USER ---
print("Please upload: 'facilities.csv', 'warehouses.csv', and 'transportation_costs.csv'")
uploaded = files.upload()

required_files = ['facilities.csv', 'warehouses.csv', 'transportation_costs.csv']
if not all(f in uploaded for f in required_files):
    print("❌ ERROR: Missing required files.")
else:
    # --- STEP 2: LOAD DATA ---
    df_facilities = pd.read_csv(io.BytesIO(uploaded['facilities.csv']))
    df_warehouses = pd.read_csv(io.BytesIO(uploaded['warehouses.csv']))
    df_transport = pd.read_csv(io.BytesIO(uploaded['transportation_costs.csv']))

    # --- STEP 3: DATA PREPARATION ---
    days = 365
    facilities = df_facilities['Facility ID'].tolist()
    warehouses = df_warehouses['Warehouse ID'].tolist()
    demands = {row['Facility ID']: row['Daily Demand'] * days for _, row in df_facilities.iterrows()}
    caps = {row['Warehouse ID']: row['Daily Capacity'] * days for _, row in df_warehouses.iterrows()}
    fixed_costs = {row['Warehouse ID']: (row['Construction Cost'] / 10) + (row['Operational Cost/Day'] * days) for _, row in df_warehouses.iterrows()}

    costs = {}
    for w in warehouses:
        costs[w] = {f: df_transport[(df_transport['Warehouse ID'] == w) & (df_transport['Facility ID'] == f)]['Cost'].values[0] for f in facilities}

    # --- STEP 4: OPTIMIZATION ---
    model = pulp.LpProblem("Campus_Logistics_CSV", pulp.LpMinimize)
    y = pulp.LpVariable.dicts("Open", warehouses, cat='Binary')
    x = pulp.LpVariable.dicts("Route", (warehouses, facilities), lowBound=0)
    model += pulp.lpSum([fixed_costs[i] * y[i] for i in warehouses]) + pulp.lpSum([costs[i][j] * x[i][j] for i in warehouses for j in facilities])

    for j in facilities:
        model += pulp.lpSum([x[i][j] for i in warehouses]) == demands[j]
    for i in warehouses:
        model += pulp.lpSum([x[i][j] for j in facilities]) <= caps[i] * y[i]
    model += pulp.lpSum([y[i] for i in warehouses]) == 2

    model.solve(pulp.PULP_CBC_CMD(msg=0))

    # --- STEP 5: TEXT OUTPUT ---
    print("\n" + "="*85)
    print("                STRATEGIC DISTRIBUTION & ROUTE LOGIC REPORT")
    print("="*85)
    total_val = pulp.value(model.objective)
    print(f"Total Annual System Cost: ${total_val:,.2f}")

    # --- STEP 5.3: NEW ROUTE COMPARISON TABLE ---
    print("\n💡 ROUTE SELECTION LOGIC (Why these routes?)")
    print("-" * 85)
    print(f"{'Department':<15} | {'Selected From':<12} | {'Unit Cost':<10} | {'Rejected Alt Cost':<20} | {'Decision Reason'}")
    print("-" * 85)

    for j in facilities:
        selected_wh = [i for i in warehouses if x[i][j].varValue > 0]
        selected_cost = min([costs[i][j] for i in selected_wh]) if selected_wh else 0

        # Find the cheapest warehouse that was NOT chosen for this facility
        rejected_costs = [costs[i][j] for i in warehouses if i not in selected_wh and y[i].varValue == 1]
        rejected_min = min(rejected_costs) if rejected_costs else "N/A"

        # Logic description
        if rejected_min == "N/A":
            reason = "Only viable hub open"
        elif selected_cost < rejected_min:
            reason = "Cheapest available route"
        else:
            reason = "Cap reached at cheaper hub"

        rej_str = f"${rejected_min:.2f}" if isinstance(rejected_min, float) else rejected_min
        print(f"{j:<15} | {str(selected_wh[0] if selected_wh else 'None'):<12} | ${selected_cost:<9.2f} | {rej_str:<20} | {reason}")
    print("-" * 85)

    # --- STEP 6: MAP REPRESENTATION (Modified to focus on selection logic) ---
    if pulp.LpStatus[model.status] == 'Optimal':
        print("\n🌐 Rendering Logistics Map...")
        w_coords = {"WH_NORTH": (34.80, -118.20), "WH_SOUTH": (34.20, -118.50), "WH_EAST": (34.50, -117.80)}
        f_coords = {"MED_CENTER": (34.60, -118.10), "ENG_BUILDING": (34.70, -118.30), "SCIENCE_HALL": (34.60, -118.60),
                    "DORM_A": (34.40, -118.40), "DORM_B": (34.30, -117.70), "LIBRARY": (34.80, -117.85)}

        m = folium.Map(location=[34.5, -118.1], zoom_start=9, tiles='CartoDB positron')

        for i in warehouses:
            color = 'red' if y[i].varValue == 1 else 'gray'
            folium.Marker(location=w_coords[i], icon=folium.Icon(color=color, icon='university', prefix='fa'),
                          popup=f"Warehouse: {i}").add_to(m)

        for j in facilities:
            folium.CircleMarker(location=f_coords[j], radius=6, color='blue', fill=True).add_to(m)
            for i in warehouses:
                if x[i][j].varValue > 0:
                    folium.PolyLine(locations=[w_coords[i], f_coords[j]], color='green', weight=4,
                                    tooltip=f"Cheaper than alternatives!").add_to(m)
        display(m)

Please upload: 'facilities.csv', 'warehouses.csv', and 'transportation_costs.csv'


Saving facilities.csv to facilities.csv
Saving transportation_costs.csv to transportation_costs.csv
Saving warehouses.csv to warehouses.csv

                STRATEGIC DISTRIBUTION & ROUTE LOGIC REPORT
Total Annual System Cost: $987,874.00

💡 ROUTE SELECTION LOGIC (Why these routes?)
-------------------------------------------------------------------------------------
Department      | Selected From | Unit Cost  | Rejected Alt Cost    | Decision Reason
-------------------------------------------------------------------------------------
MED_CENTER      | WH_NORTH     | $4.10      | $5.03                | Cheapest available route
ENG_BUILDING    | WH_NORTH     | $3.80      | $4.20                | Cheapest available route
SCIENCE_HALL    | WH_SOUTH     | $3.90      | $4.50                | Cheapest available route
DORM_A          | WH_NORTH     | $3.70      | $4.10                | Cheapest available route
DORM_B          | WH_SOUTH     | $3.68      | $4.90                | Cheapest avai